# CFG → Feature Sets → ML Prediction of logtau

Notebook này gồm:

1. Đọc metadata sau bước mapping `cfg ↔ logtau`.
2. Extract feature từ file `.cfg`.
3. Build 2 loại input:
   - `CFG only`
   - `CFG + temperature`
4. Chạy 5-fold CV + GridSearchCV cho các model:
   - Extra Trees
   - Random Forest
   - KNN
   - SVR
   - Decision Tree
   - Gradient Boosting
   - XGBoost
5. Xuất:
   - metrics từng fold
   - metrics summary
   - prediction/performance trên test theo cấu trúc nhiều cột như ảnh.


In [1]:
! pip install xgboost


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# =========================
# Cell 1 - Import
# =========================
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis

from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor


In [3]:
# =========================
# Cell 2 - Config
# =========================

metadata_file = Path(r"C:\Users\Admin\Downloads\cfg data\mapped_cfg_and_metadata\metadata_all_materials.csv")
output_dir = Path("ML_output_cfg_logtau")
output_dir.mkdir(parents=True, exist_ok=True)

# Metadata cần có tối thiểu các cột:
# material, cfg_path_mapped, cfg_file_mapped, T_cfg, logtau

TARGET_COL = "logtau"
TEMP_COL = "temperature"
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE = 0.2

# cutoff tính coordination number, đơn vị Angstrom
CUT_OFF_CN = 3.6

# bins cho distribution descriptor
CN_BINS = list(range(6, 19))          # CN = 6..18
HIST_BINS = 12                        # cho energy/force/csym/nn_distance
PERCENTILES = [5, 10, 25, 50, 75, 90, 95]


In [4]:
# =========================
# Cell 3 - Đọc CFG AtomEye
# =========================
def read_cfg_atom_eye(cfg_path):
    cfg_path = Path(cfg_path)
    with open(cfg_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    n_atoms = None
    for line in lines:
        if line.startswith("Number of particles"):
            n_atoms = int(line.split("=")[1].strip())
            break

    H = np.zeros((3, 3), dtype=float)
    for line in lines:
        m = re.match(r"H0\((\d),(\d)\)\s*=\s*([-\d\.Ee+]+)", line)
        if m:
            H[int(m.group(1))-1, int(m.group(2))-1] = float(m.group(3))

    # sau dòng auxiliary cuối cùng là data atoms
    start_idx = None
    for i, line in enumerate(lines):
        if line.startswith("auxiliary"):
            start_idx = i + 1
    if start_idx is None:
        raise ValueError(f"Không tìm thấy phần auxiliary trong {cfg_path}")

    masses, elements, frac_coords, csym, energy, forces = [], [], [], [], [], []
    i = start_idx
    while i < len(lines) - 2:
        try:
            mass = float(lines[i])
            element = lines[i+1]
            vals = list(map(float, lines[i+2].split()))
            if len(vals) >= 8:
                masses.append(mass)
                elements.append(element)
                frac_coords.append(vals[:3])
                csym.append(vals[3])
                energy.append(vals[4])
                forces.append(vals[5:8])
                i += 3
            else:
                i += 1
        except Exception:
            i += 1

    frac_coords = np.asarray(frac_coords, dtype=float)
    cart_coords = frac_coords @ H

    return {
        "n_atoms_header": n_atoms,
        "H": H,
        "volume": abs(np.linalg.det(H)),
        "masses": np.asarray(masses, dtype=float),
        "elements": np.asarray(elements),
        "frac_coords": frac_coords,
        "cart_coords": cart_coords,
        "csym": np.asarray(csym, dtype=float),
        "energy": np.asarray(energy, dtype=float),
        "forces": np.asarray(forces, dtype=float),
    }


In [5]:
# =========================
# Cell 4 - Feature helpers
# =========================
def safe_stats(x, prefix):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {}
    out = {
        f"{prefix}_mean": np.mean(x),
        f"{prefix}_std": np.std(x),
        f"{prefix}_min": np.min(x),
        f"{prefix}_max": np.max(x),
        f"{prefix}_skew": skew(x) if len(x) > 2 else 0,
        f"{prefix}_kurtosis": kurtosis(x) if len(x) > 3 else 0,
    }
    for p in PERCENTILES:
        out[f"{prefix}_p{p}"] = np.percentile(x, p)
    return out


def hist_features(x, prefix, bins=HIST_BINS, value_range=None):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {f"{prefix}_hist_{i}": 0 for i in range(bins)}
    counts, _ = np.histogram(x, bins=bins, range=value_range)
    counts = counts / max(counts.sum(), 1)
    return {f"{prefix}_hist_{i}": counts[i] for i in range(bins)}


def cn_hist_features(cn, prefix="cn"):
    cn = np.asarray(cn, dtype=int)
    total = max(len(cn), 1)
    out = {}
    for v in CN_BINS:
        out[f"{prefix}_{v}"] = np.sum(cn == v) / total
    out[f"{prefix}_lt_{CN_BINS[0]}"] = np.sum(cn < CN_BINS[0]) / total
    out[f"{prefix}_gt_{CN_BINS[-1]}"] = np.sum(cn > CN_BINS[-1]) / total
    return out


def neighbor_info(cart_coords, cutoff=CUT_OFF_CN):
    tree = cKDTree(cart_coords)
    dist, _ = tree.query(cart_coords, k=2)
    nn_dist = dist[:, 1]
    neighbors = tree.query_ball_point(cart_coords, r=cutoff)
    cn = np.array([len(n) - 1 for n in neighbors])
    return nn_dist, cn


In [8]:
# =========================
# Cell 5 - Extract feature từ 1 CFG
# 3 nhóm feature:
# 1) Basic: composition, box, density, mean/std/min/max...
# 2) Distribution: percentiles + histogram
# 3) Geometry: nearest-neighbor + coordination distribution
# =========================
def extract_cfg_features(cfg_path, cutoff_cn=CUT_OFF_CN):
    data = read_cfg_atom_eye(cfg_path)

    elements = data["elements"]
    coords = data["cart_coords"]
    energy = data["energy"]
    csym = data["csym"]
    force_mag = np.linalg.norm(data["forces"], axis=1)

    n_atoms = len(elements)
    volume = data["volume"]

    feat = {}

    # basic structural info
    feat["n_atoms"] = n_atoms
    feat["box_volume"] = volume
    feat["number_density"] = n_atoms / volume if volume > 0 else 0
    feat["box_a"] = np.linalg.norm(data["H"][0])
    feat["box_b"] = np.linalg.norm(data["H"][1])
    feat["box_c"] = np.linalg.norm(data["H"][2])

    # # composition
    # unique, counts = np.unique(elements, return_counts=True)
    # for el, c in zip(unique, counts):
    #     feat[f"comp_{el}"] = c / n_atoms * 100

    # atom-wise scalar properties: stats + histogram
    for arr, name in [(energy, "energy"), (force_mag, "force"), (csym, "csym")]:
        feat.update(safe_stats(arr, name))
        feat.update(hist_features(arr, name, bins=HIST_BINS))

    # geometry from position
    nn_dist, cn = neighbor_info(coords, cutoff=cutoff_cn)
    feat.update(safe_stats(nn_dist, "nn_dist"))
    feat.update(hist_features(nn_dist, "nn_dist", bins=HIST_BINS))
    feat.update(safe_stats(cn, "cn"))
    feat.update(cn_hist_features(cn, "cn_hist"))

    return feat


In [9]:
# =========================
# Cell 6 - Extract toàn bộ feature từ metadata
# =========================
metadata_df = pd.read_csv(metadata_file)
metadata_df.columns = metadata_df.columns.str.strip()

# chuẩn hóa tên cột nếu cần
if "cfg_path_mapped" not in metadata_df.columns and "cfg_path" in metadata_df.columns:
    metadata_df["cfg_path_mapped"] = metadata_df["cfg_path"]
if "T_cfg" not in metadata_df.columns and "temperature" in metadata_df.columns:
    metadata_df["T_cfg"] = metadata_df["temperature"]

records = []
for i, row in metadata_df.iterrows():
    try:
        feat = extract_cfg_features(row["cfg_path_mapped"], cutoff_cn=CUT_OFF_CN)
        feat["material"] = row.get("material", "unknown")
        feat["cfg_file"] = row.get("cfg_file_mapped", Path(row["cfg_path_mapped"]).name)
        feat["cfg_path"] = row["cfg_path_mapped"]
        feat[TEMP_COL] = row["T_cfg"]
        feat[TARGET_COL] = row[TARGET_COL]
        records.append(feat)
    except Exception as e:
        print("ERROR:", row.get("cfg_path_mapped"), e)

features_df = pd.DataFrame(records).fillna(0)
features_df.to_csv(output_dir / "features_all_with_metadata.csv", index=False)

print("Feature shape:", features_df.shape)
features_df.head()


Feature shape: (126, 139)


,n_atoms,box_volume,number_density,box_a,box_b,box_c,energy_mean,energy_std,energy_min,energy_max,...,cn_hist_16,cn_hist_17,cn_hist_18,cn_hist_lt_6,cn_hist_gt_18,material,cfg_file,cfg_path,temperature,logtau
0,4000,67479.291465,0.059277,40.7121,40.7121,40.7121,-5.384119,1.510739,-7.48621,-3.52664,...,0.00575,0.00000,0.00000,0.02625,0.0,Zr46Cu46Al8,Zr46Cu46Al8_638.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,638.0,15.722426
1,4000,67547.934196,0.059217,40.7259,40.7259,40.7259,-5.383971,1.507391,-7.53832,-3.33686,...,0.00575,0.00025,0.00000,0.02650,0.0,Zr46Cu46Al8,Zr46Cu46Al8_661.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,661.0,10.618194
2,4000,67550.422125,0.059215,40.7264,40.7264,40.7264,-5.382192,1.510144,-7.46329,-3.32265,...,0.00600,0.00000,0.00000,0.02375,0.0,Zr46Cu46Al8,Zr46Cu46Al8_673.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,673.0,8.450109
3,4000,67682.369797,0.059100,40.7529,40.7529,40.7529,-5.372983,1.508866,-7.51201,-3.39493,...,0.00550,0.00000,0.00025,0.02400,0.0,Zr46Cu46Al8,Zr46Cu46Al8_678.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,678.0,7.564629
4,4000,67603.180596,0.059169,40.7370,40.7370,40.7370,-5.379800,1.508646,-7.45241,-3.45033,...,0.00450,0.00025,0.00000,0.02525,0.0,Zr46Cu46Al8,Zr46Cu46Al8_680.0K.cfg,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,680.0,7.218741


In [10]:
# =========================
# Cell 7 - Build 2 feature sets
# 1) cfg only
# 2) cfg + temperature
# =========================
id_cols = ["material", "cfg_file", "cfg_path"]
non_feature_cols = id_cols + [TARGET_COL]

feature_cols_with_temp = [c for c in features_df.columns if c not in non_feature_cols]
feature_cols_cfg_only = [c for c in feature_cols_with_temp if c != TEMP_COL]

feature_sets = {
    "cfg_only": feature_cols_cfg_only,
    "cfg_plus_temperature": feature_cols_with_temp,
}

for name, cols in feature_sets.items():
    out = features_df[id_cols + cols + [TARGET_COL]].copy()
    out.to_csv(output_dir / f"dataset_{name}.csv", index=False)
    print(name, out.shape)


cfg_only (126, 138)
cfg_plus_temperature (126, 139)


In [13]:
# =========================
# Cell 8 - Model + hyperparameter grids
# =========================
def get_models_and_grids():
    models = {
        "Extra Trees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "KNN": KNeighborsRegressor(),
        "SVR": SVR(),
        "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }

    grids = {
        "Extra Trees": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "Random Forest": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "KNN": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights": ["uniform", "distance"],
        },
        "SVR": {
            "model__C": [1, 10, 100],
            "model__epsilon": [0.01, 0.1],
            "model__kernel": ["rbf"],
        },
        "Decision Tree": {
            "model__max_depth": [None, 3, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "Gradient Boosting": {
            "model__n_estimators": [100, 300],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [2, 3],
        },
            "XGBoost": {
            "model__n_estimators": [100, 300],
            "model__max_depth": [2, 3, 5],
            "model__learning_rate": [0.03, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
            }
        }
    return models, grids


def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


In [20]:
# =========================
# Cell 9 - Metric functions
# =========================
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_true, y_pred),
    }


def clean_params(params):
    return {k.replace("model__", ""): v for k, v in params.items()}


In [24]:
# =========================
# Cell 10 - 5-fold CV + GridSearchCV + hold-out test
# =========================
def run_ml_for_feature_set(features_df, feature_cols, set_name):
    X = features_df[feature_cols].copy()
    y = features_df[TARGET_COL].astype(float).values
    meta = features_df[["material", "cfg_file", "cfg_path", TEMP_COL]].copy()

    X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
        X, y, meta, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    models, grids = get_models_and_grids()
    outer_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    fold_records = []
    summary_records = []
    performance_df = pd.DataFrame(index=np.arange(len(y_test)))

    performance_df[("Info", "material")] = meta_test["material"].values
    performance_df[("Info", "cfg_file")] = meta_test["cfg_file"].values
    performance_df[("Info", "temperature")] = meta_test[TEMP_COL].values

    for model_name, model in models.items():
        print(f"[{set_name}] Running {model_name}...")
        pipe = make_pipeline(model)
        grid = grids[model_name]

        y_test_pred_folds = []
        fold_best_params = []

        for fold, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train), start=1):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            search = GridSearchCV(
                pipe, grid, cv=3, scoring="r2", n_jobs=-1, error_score="raise"
            )
            search.fit(X_tr, y_tr)
            best = search.best_estimator_

            pred_tr = best.predict(X_tr)
            pred_val = best.predict(X_val)

            m_tr = calc_metrics(y_tr, pred_tr)
            m_val = calc_metrics(y_val, pred_val)

            fold_records.append({
                "feature_set": set_name,
                "model": model_name,
                "fold": fold,
                "best_params": json.dumps(clean_params(search.best_params_)),
                "train_r2": m_tr["r2"],
                "train_rmse": m_tr["rmse"],
                "train_mae": m_tr["mae"],
                "val_r2": m_val["r2"],
                "val_rmse": m_val["rmse"],
                "val_mae": m_val["mae"],
            })

            fold_best_params.append(search.best_params_)
            y_test_pred_folds.append(best.predict(X_test))

        # final search on full train set
        final_search = GridSearchCV(
            pipe, grid, cv=5, scoring="r2", n_jobs=-1, error_score="raise"
        )
        final_search.fit(X_train, y_train)
        final_model = final_search.best_estimator_

        y_pred_train = final_model.predict(X_train)
        y_pred_test = final_model.predict(X_test)

        train_m = calc_metrics(y_train, y_pred_train)
        test_m = calc_metrics(y_test, y_pred_test)

        summary_records.append({
            "feature_set": set_name,
            "model": model_name,
            "best_params": json.dumps(clean_params(final_search.best_params_)),
            "train_r2": train_m["r2"],
            "train_rmse": train_m["rmse"],
            "train_mae": train_m["mae"],
            "test_r2": test_m["r2"],
            "test_rmse": test_m["rmse"],
            "test_mae": test_m["mae"],
        })

        performance_df[(model_name, "y_true")] = y_test
        performance_df[(model_name, "y_pred")] = y_pred_test

    fold_df = pd.DataFrame(fold_records)
    summary_df = pd.DataFrame(summary_records).sort_values(
        ["test_r2", "test_rmse", "test_mae"], ascending=[False, True, True]
    )

    # lưu CSV thường
    fold_df.to_csv(output_dir / f"metrics_folds_{set_name}.csv", index=False)
    summary_df.to_csv(output_dir / f"metrics_summary_{set_name}.csv", index=False)

    # CSV không hỗ trợ multi-header đẹp bằng Excel, nhưng vẫn lưu 2 dòng header
    performance_df.to_csv(output_dir / f"test_performance_{set_name}.csv", index=False)

    return fold_df, summary_df, performance_df


In [25]:
# =========================
# Cell 11 - Run cho 2 input sets
# =========================
all_fold_metrics = []
all_summary = []

results = {}
for set_name, cols in feature_sets.items():
    fold_df, summary_df, perf_df = run_ml_for_feature_set(features_df, cols, set_name)
    all_fold_metrics.append(fold_df)
    all_summary.append(summary_df)
    results[set_name] = {"fold": fold_df, "summary": summary_df, "performance": perf_df}

all_fold_metrics = pd.concat(all_fold_metrics, ignore_index=True)
all_summary = pd.concat(all_summary, ignore_index=True)

all_fold_metrics.to_csv(output_dir / "metrics_folds_all_feature_sets.csv", index=False)
all_summary.to_csv(output_dir / "metrics_summary_all_feature_sets.csv", index=False)

print("Done. Saved outputs to:", output_dir)
all_summary


[cfg_only] Running Extra Trees...
[cfg_only] Running Random Forest...
[cfg_only] Running KNN...
[cfg_only] Running SVR...
[cfg_only] Running Decision Tree...
[cfg_only] Running Gradient Boosting...
[cfg_plus_temperature] Running Extra Trees...
[cfg_plus_temperature] Running Random Forest...
[cfg_plus_temperature] Running KNN...
[cfg_plus_temperature] Running SVR...
[cfg_plus_temperature] Running Decision Tree...
[cfg_plus_temperature] Running Gradient Boosting...
Done. Saved outputs to: ML_output_cfg_logtau


,feature_set,model,best_params,train_r2,train_rmse,train_mae,test_r2,test_rmse,test_mae
0,cfg_only,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",0.999960,4.698775e-02,3.885498e-02,0.918390,2.413278,1.759147
1,cfg_only,Extra Trees,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",1.000000,3.336554e-14,2.603862e-14,0.882712,2.893101,2.008535
2,cfg_only,Random Forest,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.976766,1.131704e+00,7.878705e-01,0.842333,3.354335,2.187322
3,cfg_only,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",0.999998,9.990840e-03,9.970364e-03,0.822801,3.556045,2.639238
4,cfg_only,Decision Tree,"{""max_depth"": 3, ""min_samples_leaf"": 1}",0.899223,2.356948e+00,1.567853e+00,0.763809,4.105525,3.344496
5,cfg_only,KNN,"{""n_neighbors"": 7, ""weights"": ""distance""}",1.000000,4.061449e-07,1.884874e-07,0.623127,5.186016,3.590859
6,cfg_plus_temperature,Gradient Boosting,"{""learning_rate"": 0.03, ""max_depth"": 2, ""n_est...",0.999081,2.251311e-01,1.771115e-01,0.948007,1.926237,1.218650
7,cfg_plus_temperature,Extra Trees,"{""max_depth"": 10, ""min_samples_leaf"": 2, ""n_es...",0.999388,1.836764e-01,1.179090e-01,0.931294,2.214284,1.370364
8,cfg_plus_temperature,Random Forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.985933,8.805716e-01,5.924217e-01,0.898151,2.695971,1.619095
9,cfg_plus_temperature,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",0.999998,9.957148e-03,9.917821e-03,0.831643,3.466193,2.568487


In [26]:
# =========================
# Cell 12 - Xem nhanh kết quả tốt nhất
# =========================
summary_view = all_summary.copy()
for c in ["train_r2", "test_r2"]:
    summary_view[c] = summary_view[c] * 100

num_cols = ["train_r2", "train_rmse", "train_mae", "test_r2", "test_rmse", "test_mae"]
summary_view[num_cols] = summary_view[num_cols].round(2)

summary_view.sort_values(["test_r2", "test_rmse", "test_mae"], ascending=[False, True, True])


,feature_set,model,best_params,train_r2,train_rmse,train_mae,test_r2,test_rmse,test_mae
6,cfg_plus_temperature,Gradient Boosting,"{""learning_rate"": 0.03, ""max_depth"": 2, ""n_est...",99.91,0.23,0.18,94.80,1.93,1.22
7,cfg_plus_temperature,Extra Trees,"{""max_depth"": 10, ""min_samples_leaf"": 2, ""n_es...",99.94,0.18,0.12,93.13,2.21,1.37
0,cfg_only,Gradient Boosting,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",100.00,0.05,0.04,91.84,2.41,1.76
8,cfg_plus_temperature,Random Forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",98.59,0.88,0.59,89.82,2.70,1.62
1,cfg_only,Extra Trees,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",100.00,0.00,0.00,88.27,2.89,2.01
2,cfg_only,Random Forest,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",97.68,1.13,0.79,84.23,3.35,2.19
9,cfg_plus_temperature,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",100.00,0.01,0.01,83.16,3.47,2.57
3,cfg_only,SVR,"{""C"": 100, ""epsilon"": 0.01, ""kernel"": ""rbf""}",100.00,0.01,0.01,82.28,3.56,2.64
4,cfg_only,Decision Tree,"{""max_depth"": 3, ""min_samples_leaf"": 1}",89.92,2.36,1.57,76.38,4.11,3.34
10,cfg_plus_temperature,KNN,"{""n_neighbors"": 5, ""weights"": ""distance""}",100.00,0.00,0.00,66.69,4.88,3.02
